<a href="https://colab.research.google.com/github/hari07-create/WorldCup_Analysis/blob/main/world_cup_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
excel_file = "world_cup_results.xlsx"

df_matches = pd.read_excel(excel_file, sheet_name="WorldCupMatches")
df_summary = pd.read_excel(excel_file, sheet_name="WorldCups")

In [ ]:
print(f"Matches: {df_matches.shape}, Summary: {df_summary.shape}")

In [ ]:
def clean_country_names(name):
    if pd.isna(name):
        return name
    name = str(name).strip()
    if name in ["Germany FR", "West Germany"]:
        return "Germany"
    if name == "Soviet Union":
        return "Russia"
    return name

In [ ]:
df_matches["HomeTeam"] = df_matches["HomeTeam"].apply(clean_country_names)
df_matches["AwayTeam"] = df_matches["AwayTeam"].apply(clean_country_names)
df_summary["Winner"] = df_summary["Winner"].apply(clean_country_names)

In [ ]:
def clean_attendance(val):
    val_str = str(val).strip().replace('.', '')
    if not val_str or val_str == 'nan':
        return np.nan
    v = float(val_str)
    # Fixes compressed scientific notations/short strings in early years
    if v < 5000:
        if v < 1000:
            v *= 1000
        else:
            v *= 100
    elif v == 81981.0: # Explicit fix for 1958 scaling anomaly
        v *= 10
    return v

In [ ]:
df_summary["Attendance"] = df_summary["Attendance"].apply(clean_attendance)

In [ ]:
df_matches["Total Goals"] = df_matches["HomeGoals"] + df_matches["AwayGoals"]

In [ ]:
def determine_match_winner(row):
    if row["HomeGoals"] > row["AwayGoals"]:
        return row["HomeTeam"]
    elif row["AwayGoals"] > row["HomeGoals"]:
        return row["AwayTeam"]
    else:
        # Check 'Observation' string for penalty shootout or extra-time designations
        obs = str(row["Observation"]).lower()
        if "win" in obs:
            if str(row["HomeTeam"]).lower() in obs:
                return row["HomeTeam"]
            elif str(row["AwayTeam"]).lower() in obs:
                return row["AwayTeam"]
        return "Draw"

df_matches["Match Winner"] = df_matches.apply(determine_match_winner, axis=1)

In [ ]:
plt.figure(figsize=(10, 5))
goals_per_year = df_matches.groupby("Year")["Total Goals"].mean().reset_index()

sns.lineplot(data=goals_per_year, x="Year", y="Total Goals", marker="o", color="#1f77b4", linewidth=2.5)
plt.title("Average Goals Scored Per Match Over Time", weight='bold', pad=15)
plt.ylabel("Avg Goals per Match")
plt.xlabel("Tournament Year")
plt.xticks(goals_per_year["Year"].unique(), rotation=45)

plt.tight_layout()
plt.savefig("1_goals_trend_over_time.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
home_g = df_matches.groupby("HomeTeam")["HomeGoals"].sum().reset_index().rename(columns={"HomeTeam": "Team", "HomeGoals": "Goals"})
away_g = df_matches.groupby("AwayTeam")["AwayGoals"].sum().reset_index().rename(columns={"AwayTeam": "Team", "AwayGoals": "Goals"})
all_g = pd.concat([home_g, away_g]).groupby("Team")["Goals"].sum().reset_index()
top_scorers = all_g.sort_values(by="Goals", ascending=False).head(10)

sns.barplot(data=top_scorers, x="Goals", y="Team", palette="viridis")
plt.title("Top 10 All-Time Highest Scoring Countries (1930-2014)", weight='bold', pad=15)
plt.xlabel("Total Goals Scored")
plt.ylabel("")

plt.tight_layout()
plt.savefig("2_top_scoring_countries.png", dpi=300)
plt.show()

In [ ]:
df_matches.to_csv('WorldCupMatches_cleaned.csv', index=False)
df_summary.to_csv('WorldCups_summary_cleaned.csv', index=False)